# lab 8: un agent cu instrumente și o bază de date

scopul lucrării: să facem un agent mic, care lucrează cu o bază de date sqlite a unei biblioteci.

un agent este un model de limbaj care poate folosi **instrumente** (tools). instrumentele sunt funcții python obișnuite. modelul nu le rulează singur, el doar spune ce funcție vrea și cu ce argumente. noi rulăm funcția și îi dăm rezultatul înapoi.

agentul nostru are 3 instrumente:

1. `cauta_carti` caută cărți după titlu, autor sau gen
2. `imprumuta_carte` împrumută o carte unui cititor
3. `carti_imprumutate` arată ce cărți are un cititor

sarcina agentului: să primească o cerere de la un cititor, de exemplu "vreau să împrumut o carte de Eminescu", să găsească cartea în bază și să facă împrumutul.

In [2]:
import json
import re
import sqlite3
import unicodedata
import torch
from datetime import date
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print(device)

cuda


## 1. baza de date

facem un fișier sqlite cu două tabele:

* `carti`: id, titlu, autor, gen și câte exemplare libere sunt
* `imprumuturi`: ce carte, ce cititor și în ce zi

ștergem tabelele la început, ca la fiecare rulare să pornim de la aceeași bază.

In [3]:
db = sqlite3.connect("biblioteca.db")
db.executescript("""
DROP TABLE IF EXISTS imprumuturi;
DROP TABLE IF EXISTS carti;
CREATE TABLE carti (
    id INTEGER PRIMARY KEY,
    titlu TEXT,
    autor TEXT,
    gen TEXT,
    exemplare INTEGER
);
CREATE TABLE imprumuturi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    carte_id INTEGER,
    cititor TEXT,
    data TEXT
);
""")

carti = [
    (1, "Poezii", "Mihai Eminescu", "poezie", 2),
    (2, "Luceafărul", "Mihai Eminescu", "poezie", 0),
    (3, "Amintiri din copilărie", "Ion Creangă", "proză", 3),
    (4, "Moara cu noroc", "Ioan Slavici", "proză", 1),
    (5, "Enigma Otiliei", "George Călinescu", "roman", 2),
    (6, "Maitreyi", "Mircea Eliade", "roman", 1),
    (7, "O scrisoare pierdută", "Ion Luca Caragiale", "teatru", 2),
]
db.executemany("INSERT INTO carti VALUES (?, ?, ?, ?, ?)", carti)
db.commit()

for row in db.execute("SELECT * FROM carti"):
    print(row)

(1, 'Poezii', 'Mihai Eminescu', 'poezie', 2)
(2, 'Luceafărul', 'Mihai Eminescu', 'poezie', 0)
(3, 'Amintiri din copilărie', 'Ion Creangă', 'proză', 3)
(4, 'Moara cu noroc', 'Ioan Slavici', 'proză', 1)
(5, 'Enigma Otiliei', 'George Călinescu', 'roman', 2)
(6, 'Maitreyi', 'Mircea Eliade', 'roman', 1)
(7, 'O scrisoare pierdută', 'Ion Luca Caragiale', 'teatru', 2)


## 2. instrumentele

fiecare instrument este o funcție python. comentariul de sub nume (docstring) este important: din el modelul află ce face funcția și ce argumente are.

toate funcțiile întorc text, pentru că modelul citește doar text.

modelul este mic și scrie argumentele cu greșeli: "romane" în loc de "roman", "Creanga" fără diacritice. de aceea căutarea este tolerantă: scoate diacriticele și compară doar primele 5 litere din fiecare cuvânt. un instrument bun trebuie să suporte astfel de greșeli.

In [4]:
def simplu(text):
    # lower case, without diacritics: "Creangă" -> "creanga"
    text = unicodedata.normalize("NFD", text.lower())
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")


def cauta_carti(text: str) -> str:
    """
    Caută cărți în bibliotecă după titlu, autor sau gen.

    Args:
        text: ce căutăm, de exemplu "Eminescu", "roman" sau "Amintiri din copilărie"
    """
    # compare only the first 5 letters of each word: "romane" finds "roman"
    words = [simplu(w)[:5] for w in re.findall(r"\w+", text) if len(w) >= 4]
    found = []
    for i, titlu, autor, gen, n in db.execute("SELECT * FROM carti"):
        row = simplu(f"{titlu} {autor} {gen}")
        if any(w in row for w in words):
            found.append(f"id={i}, '{titlu}' de {autor}, gen {gen}, exemplare libere: {n}")
    return "\n".join(found) if found else "Nu am găsit nicio carte."


def imprumuta_carte(carte_id: int, cititor: str) -> str:
    """
    Împrumută o carte unui cititor, dacă există exemplare libere.

    Args:
        carte_id: id-ul cărții din bibliotecă
        cititor: numele cititorului
    """
    row = db.execute("SELECT titlu, exemplare FROM carti WHERE id = ?", (carte_id,)).fetchone()
    if row is None:
        return f"Nu există carte cu id={carte_id}."
    titlu, exemplare = row
    if exemplare == 0:
        return f"'{titlu}' nu are exemplare libere."
    db.execute("UPDATE carti SET exemplare = exemplare - 1 WHERE id = ?", (carte_id,))
    db.execute("INSERT INTO imprumuturi (carte_id, cititor, data) VALUES (?, ?, ?)",
               (carte_id, cititor, str(date.today())))
    db.commit()
    return f"Gata: '{titlu}' a fost împrumutată de {cititor}."


def carti_imprumutate(cititor: str) -> str:
    """
    Arată cărțile împrumutate de un cititor.

    Args:
        cititor: numele cititorului
    """
    rows = db.execute(
        "SELECT c.titlu, i.data FROM imprumuturi i JOIN carti c ON c.id = i.carte_id "
        "WHERE i.cititor = ?", (cititor,)).fetchall()
    if not rows:
        return f"{cititor} nu are cărți împrumutate."
    return "\n".join(f"'{t}', din {d}" for t, d in rows)


TOOLS = {f.__name__: f for f in [cauta_carti, imprumuta_carte, carti_imprumutate]}

încercăm instrumentele de mână, fără model, ca să vedem că merg.

In [5]:
print(cauta_carti("Eminescu"))
print(cauta_carti("romane"))
print(cauta_carti("Ion Creanga"))
print(carti_imprumutate("Ion"))

id=1, 'Poezii' de Mihai Eminescu, gen poezie, exemplare libere: 2
id=2, 'Luceafărul' de Mihai Eminescu, gen poezie, exemplare libere: 0
id=5, 'Enigma Otiliei' de George Călinescu, gen roman, exemplare libere: 2
id=6, 'Maitreyi' de Mircea Eliade, gen roman, exemplare libere: 1
id=3, 'Amintiri din copilărie' de Ion Creangă, gen proză, exemplare libere: 3
Ion nu are cărți împrumutate.


## 3. modelul

folosim `Qwen2.5-3B-Instruct`. el știe să ceară instrumente: scrie un json cu numele funcției și argumentele, între etichetele `<tool_call>` și `</tool_call>`.

am încercat întâi modelul mai mic din lab 6, `Qwen2.5-1.5B-Instruct`. el găsea cartea, dar se oprea după căutare și nu mai făcea împrumutul. pentru un agent care face mai mulți pași la rând, modelul de 3B merge mult mai bine.

`device_map="auto"` pune modelul pe gpu, iar dacă nu încape tot, mută o parte în memoria ram. în colab, pe un t4, modelul încape tot pe gpu.

In [6]:
LLM = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(
    LLM, torch_dtype=torch.float16 if device == "cuda" else torch.float32, device_map="auto")
llm.eval()

def model_reply(messages):
    # tools=... puts the description of our functions into the prompt
    text = tok.apply_chat_template(messages, tools=list(TOOLS.values()),
                                   tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**ids, max_new_tokens=300, do_sample=False)
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


să vedem cum arată prompt-ul. modelul primește descrierea instrumentelor, făcută automat din docstring-uri.

In [7]:
demo = tok.apply_chat_template([{"role": "user", "content": "salut"}],
                               tools=list(TOOLS.values()), tokenize=False)
print(demo[:1500])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "cauta_carti", "description": "Caută cărți în bibliotecă după titlu, autor sau gen.", "parameters": {"type": "object", "properties": {"text": {"type": "string", "description": "ce căutăm, de exemplu \"Eminescu\", \"roman\" sau \"Amintiri din copilărie\""}}, "required": ["text"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "imprumuta_carte", "description": "Împrumută o carte unui cititor, dacă există exemplare libere.", "parameters": {"type": "object", "properties": {"carte_id": {"type": "integer", "description": "id-ul cărții din bibliotecă"}, "cititor": {"type": "string", "description": "numele cititorului"}}, "required": ["carte_id", "cititor"]}, "return": {"type": "str

## 4. bucla agentului

agentul lucrează în pași:

1. modelul primește cererea și răspunde
2. dacă în răspuns este `<tool_call>`, rulăm funcția cerută și punem rezultatul în conversație
3. repetăm până când modelul răspunde fără să mai ceară un instrument
4. ne oprim după cel mult 5 pași, ca să nu intre într-o buclă fără sfârșit

dacă modelul cere un instrument care nu există sau dă argumente greșite, nu oprim programul. îi trimitem eroarea ca text, ca să se poată corecta.

instrucțiunea de sistem o scriem în engleză. am încercat-o întâi în română, dar modelul mic aproape nu folosea instrumentele și inventa răspunsuri. cererile cititorilor rămân în română.

In [8]:
# in english: the small model follows english instructions better
SYSTEM = ("You are a library assistant. Always use the tools to search books and to borrow them. "
          "To borrow a book, first call cauta_carti to find its id, then call imprumuta_carte. "
          "Answer briefly in Romanian.")

def find_calls(text):
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", text, re.S):
        try:
            calls.append(json.loads(block))
        except json.JSONDecodeError:
            pass                                  # bad json, skip it
    return calls

def agent(cerere, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": cerere}]
    print("CERERE:", cerere)
    for step in range(max_steps):
        reply = model_reply(messages)
        calls = find_calls(reply)
        if not calls:
            print("AGENT:", reply.strip())
            return reply
        messages.append({"role": "assistant", "content": reply})
        for call in calls:
            name, args = call.get("name"), call.get("arguments", {})
            try:
                result = TOOLS[name](**args)
            except Exception as e:                # wrong tool or wrong arguments
                result = f"Eroare: {e}"
            print(f"  pas {step + 1}: {name}({args})")
            print("   ->", result.replace("\n", "\n      "))
            messages.append({"role": "tool", "name": name, "content": result})
    print("AGENT: m-am oprit după", max_steps, "pași")

## 5. sarcinile agentului

### a. o căutare simplă

In [9]:
agent("Ce romane aveți în bibliotecă?")

CERERE: Ce romane aveți în bibliotecă?


  pas 1: cauta_carti({'text': 'roman'})
   -> id=5, 'Enigma Otiliei' de George Călinescu, gen roman, exemplare libere: 2
      id=6, 'Maitreyi' de Mircea Eliade, gen roman, exemplare libere: 1


AGENT: Avem două exemplare ale cărții "Enigma Otiliei" de George Călinescu și una pentru "Maitreyi" de Mircea Eliade. Am găsit cărți din genul roman.


'Avem două exemplare ale cărții "Enigma Otiliei" de George Călinescu și una pentru "Maitreyi" de Mircea Eliade. Am găsit cărți din genul roman.'

### b. sarcina principală: un împrumut

agentul trebuie să facă doi pași singur: să caute cartea ca să afle id-ul, apoi să o împrumute.

In [10]:
agent("Sunt Ana Popescu. Vreau să împrumut Amintiri din copilărie de Ion Creangă.")

CERERE: Sunt Ana Popescu. Vreau să împrumut Amintiri din copilărie de Ion Creangă.


  pas 1: cauta_carti({'text': 'Amintiri din copilărie Creangă'})
   -> id=3, 'Amintiri din copilărie' de Ion Creangă, gen proză, exemplare libere: 3


  pas 2: imprumuta_carte({'carte_id': 3, 'cititor': 'Ana Popescu'})
   -> Gata: 'Amintiri din copilărie' a fost împrumutată de Ana Popescu.


AGENT: Cartea 'Amintiri din copilărie' de Ion Creangă a fost cu succes împrumutată.


"Cartea 'Amintiri din copilărie' de Ion Creangă a fost cu succes împrumutată."

### c. o carte fără exemplare libere

"Luceafărul" are 0 exemplare. agentul trebuie să spună că nu se poate.

In [11]:
agent("Sunt Ana Popescu. Vreau să împrumut Luceafărul de Eminescu.")

CERERE: Sunt Ana Popescu. Vreau să împrumut Luceafărul de Eminescu.


  pas 1: cauta_carti({'text': 'Luceafărul Eminescu'})
   -> id=1, 'Poezii' de Mihai Eminescu, gen poezie, exemplare libere: 2
      id=2, 'Luceafărul' de Mihai Eminescu, gen poezie, exemplare libere: 0


AGENT: Această carte 'Luceafărul' nu este în stoc. Vă recomand să consultați alte cărți din același gen.


"Această carte 'Luceafărul' nu este în stoc. Vă recomand să consultați alte cărți din același gen."

### d. ce cărți are cititorul

In [12]:
agent("Ce cărți am împrumutate? Numele meu este Ana Popescu.")

CERERE: Ce cărți am împrumutate? Numele meu este Ana Popescu.


  pas 1: carti_imprumutate({'cititor': 'Ana Popescu'})
   -> 'Amintiri din copilărie', din 2026-11-02


AGENT: Ai împrumutat cartea "Amintiri din copilărie" de 2026-11-02.


'Ai împrumutat cartea "Amintiri din copilărie" de 2026-11-02.'

## 6. verificăm baza de date

nu ne bazăm doar pe ce spune agentul. ne uităm direct în tabele.

In [13]:
print("imprumuturi:")
for row in db.execute("SELECT * FROM imprumuturi"):
    print(row)
print("\ncarti:")
for row in db.execute("SELECT id, titlu, exemplare FROM carti"):
    print(row)

imprumuturi:
(1, 3, 'Ana Popescu', '2026-11-02')

carti:
(1, 'Poezii', 2)
(2, 'Luceafărul', 0)
(3, 'Amintiri din copilărie', 2)
(4, 'Moara cu noroc', 1)
(5, 'Enigma Otiliei', 2)
(6, 'Maitreyi', 1)
(7, 'O scrisoare pierdută', 2)


## concluzii

* agentul a rezolvat toate cele 4 cereri. la căutare a folosit un singur instrument. la împrumut a făcut singur doi pași: a căutat cartea, a aflat că are id=3, apoi a chemat `imprumuta_carte` cu id-ul și numele cititorului.
* baza de date confirmă împrumutul: în tabelul `imprumuturi` a apărut rândul (1, 3, 'Ana Popescu', '2026-11-02'), iar "Amintiri din copilărie" a scăzut de la 3 la 2 exemplare.
* la "Luceafărul" agentul a văzut că sunt 0 exemplare și nu a mai chemat `imprumuta_carte`. a spus corect că nu se poate. chiar dacă ar fi chemat-o, funcția verifică singură stocul, deci baza nu se putea strica.
* modelul contează mult. cu `Qwen2.5-1.5B-Instruct` și cu instrucțiunea în română, modelul aproape nu folosea instrumentele și inventa răspunsuri. cu instrucțiunea în engleză le folosea, dar se oprea după căutare și nu făcea împrumutul. abia `Qwen2.5-3B-Instruct` a dus sarcina până la capăt.
* instrumentele trebuie să fie tolerante. modelul scria "romane" în loc de "roman" și "Creanga" fără diacritice. căutarea fără diacritice și după primele 5 litere a rezolvat asta.
* regulile importante stau în cod, nu în prompt: verificarea exemplarelor, limita de 5 pași și prinderea erorilor. modelul poate greși, dar baza de date rămâne corectă.
* româna modelului nu este perfectă (de exemplu "ai împrumutat cartea de 2026-11-02" în loc de "din"), dar informația este corectă.